# Accessibility Platform — Colab runner OSRM (ARG / MEX / BRA)

Notebook empaquetado para correr el indicador OSRM de accesibilidad escolar para los 3 países que no entran en una laptop de 16 GB. Steps 09b (matrices OSRM) + 10b (agregación SCL) sobre inputs ya producidos por steps 01–07 en local.

**OSRM se compila desde fuente.** conda-forge no tiene `osrm-backend` y Colab no permite Docker/udocker (lo marca como uso prohibido). La celda de setup compila `osrm-backend v5.27.1` (gcc-12 + libtbb-dev) y **cachea los binarios en Drive** (`inputs/bin/osrm/`): la 1ra vez ~20-30 min, después se restauran en segundos.

**Una pasada por modo (`--all-combos`):** 09b rutea cada celda **una sola vez por modo** y deriva los 6 parquets (2 sectores × 3 niveles) de esa única respuesta OSRM — ~6× menos búsquedas de red que el loop por-combo anterior. La salida es **bit-idéntica** al método por-combo (validado en `tests/test_osrm_all_combos_equivalence.py`). El `Total` del SCL lo deriva 10b como `min(public, private)`, igual que FMM. Un sector sin escuela georeferenciada se saltea solo (no aborta).

**Prerequisitos**
- Carpeta `idb_access/` en Google Drive con la estructura del `README.md` adyacente. En la celda 1 ajustá `DRIVE_ROOT` si tu carpeta no está en `MyDrive/idb_access` (admite rutas con espacios, p. ej. `MyDrive/Colab Notebooks/datasets/idb_access`).
- Runtime Colab Pro con **High-RAM** (≥25 GB para ARG/MEX; Pro+ con 51 GB para BRA)
- El repo debe estar pusheado con `--all-combos` + el split por sector + la migración a OSRM-desde-fuente (rama `step09-travel-times`); el notebook lo toma con `git pull`.
- Tiempos esperados con `--all-combos` (aprox., depende de los cores del runtime): ARG ~2-3 h · MEX ~2-3 h · BRA ~6-10 h (+ ~20-30 min de compilación la 1ra vez). OSRM es CPU-bound: el límite lo ponen los vCPU de Colab, no la concurrencia del cliente.

**Nota `--all-combos`:** rutear una vez por modo requiere tener en RAM los índices de las K escuelas más cercanas por celda de los 6 combos a la vez. Para MEX/BRA en High-RAM entra bien; si el runtime es chico y ves OOM, bajá `--k` (p. ej. 30) — sigue cubriendo la más cercana en la práctica.

**Para usar:** ajustá `COUNTRY` en la celda siguiente, andá a `Runtime → Change runtime type → High-RAM`, después `Runtime → Run all`.

**Resume (BRA, run largo):** si la sesión se corta, volvé a correr — los parquets de matrices ya escritos se saltean (las celdas no pasan `--overwrite`) y sigue donde quedó. Los grafos OSRM y los binarios compilados quedan cacheados en Drive entre sesiones.

In [ ]:
# === Parámetros — ajustar antes de correr ===
COUNTRY = "ARG"   # ARG | MEX | BRA
# Carpeta idb_access en tu Drive (admite espacios). Cambiala si la movés.
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks/datasets/idb_access"
REPO_URL = "https://github.com/AngelaLop/accessibility_platform.git"
REPO_BRANCH = "step09-travel-times"  # cambiar a `main` cuando se mergee

GEOFABRIK_PATH = {
    "ARG": "south-america/argentina",
    "MEX": "north-america/mexico",
    "BRA": "south-america/brazil",
}[COUNTRY]
print(f"Configurado para {COUNTRY} ({GEOFABRIK_PATH})")

In [ ]:
# === Chequeo de runtime ===
import os
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
cpus = os.cpu_count()
print(f"RAM: {ram_gb:.1f} GB | CPUs: {cpus}")
if COUNTRY == "BRA" and ram_gb < 25:
    raise RuntimeError(
        "BRA necesita >=25 GB de RAM. Cambiá a un runtime High-RAM "
        "(Runtime → Change runtime type → High-RAM)."
    )
if ram_gb < 12:
    raise RuntimeError(f"RAM insuficiente ({ram_gb:.1f} GB). Usá High-RAM runtime.")


In [ ]:
# === Mount de Google Drive ===
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path(DRIVE_ROOT)
assert DRIVE.exists(), (
    f"No encuentro {DRIVE_ROOT}. Creá la carpeta en tu Drive con la "
    f"estructura descripta en notebooks/README.md y subí los inputs."
)
print(f"Drive root OK: {DRIVE}")


In [ ]:
# === Clone o actualización del repo ===
REPO = Path("/content/repo")
if REPO.exists():
    !cd {REPO} && git fetch && git checkout {REPO_BRANCH} && git pull --ff-only
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO}
!cd {REPO} && git log --oneline -3


In [ ]:
# === Setup: uv (deps Python) + OSRM compilado desde fuente (cacheado en Drive) ===
# 1) uv: instalador de deps pinneadas desde uv.lock (commit 73a1548). El binario
#    queda en /usr/local/bin (no en /root/.local/bin) — lo llamamos vía PATH.
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"/usr/local/bin:/root/.local/bin:/root/.cargo/bin:{os.environ['PATH']}"
!cd {REPO} && uv sync --frozen --python 3.13 2>&1 | tail -10

# 2) OSRM: conda-forge NO tiene osrm-backend y Colab no permite Docker/udocker.
#    Compilamos osrm-backend desde fuente (legítimo: solo es compilar C++) y
#    cacheamos los binarios + perfiles en Drive. La 1ra vez: ~20-30 min. Las
#    siguientes sesiones se restauran en segundos. Receta Ubuntu 22.04: gcc-12
#    (gcc-11 falla) + libtbb-dev + tag v5.27.1.
import shutil
OSRM_TAG = "v5.27.1"
OSRM_CACHE = DRIVE / "inputs" / "bin" / "osrm"     # binarios + perfiles en Drive
OSRM_BINS = ["osrm-extract", "osrm-partition", "osrm-customize", "osrm-routed", "osrm-contract"]
PROFILES = Path("/content/osrm_profiles")          # perfiles Lua (local, sin espacios)

# Deps de runtime/compilación (siempre: proveen las .so que los binarios linkean)
print("Instalando dependencias del sistema (apt)...")
!apt-get -qq update 2>&1 | tail -1
!apt-get -qq install -y build-essential gcc-12 g++-12 cmake pkg-config \
    libboost-all-dev libtbb-dev lua5.4 liblua5.4-dev libbz2-dev libzip-dev \
    libxml2-dev expat libexpat1-dev zlib1g-dev 2>&1 | tail -2

if (OSRM_CACHE / "osrm-routed").exists():
    print(f"OSRM {OSRM_TAG} cacheado en Drive — restaurando binarios...")
    for b in OSRM_BINS:
        src = OSRM_CACHE / b
        if src.exists():
            shutil.copy(src, f"/usr/local/bin/{b}")
            os.chmod(f"/usr/local/bin/{b}", 0o755)
    shutil.copytree(OSRM_CACHE / "profiles", PROFILES, dirs_exist_ok=True)
else:
    print(f"Compilando osrm-backend {OSRM_TAG} (1ra vez — paciencia, ~20-30 min)...")
    !rm -rf /content/osrm-backend
    !git clone --branch {OSRM_TAG} --depth 1 https://github.com/Project-OSRM/osrm-backend.git /content/osrm-backend
    # OSRM mete -Werror (no como literal en los CMake, no se puede sed-ear) y con
    # gcc-12 + Boost de Colab alguna deprecation lo vuelve fatal. Solución robusta:
    # un wrapper del compilador que agrega -w (apaga TODAS las advertencias) al
    # final → sin advertencias, -Werror no tiene nada que convertir en error.
    for name, real in [("cc-nowarn", "gcc-12"), ("cxx-nowarn", "g++-12")]:
        wp = Path(f"/usr/local/bin/{name}")
        wp.write_text(f'#!/bin/bash\nexec {real} "$@" -w\n')
        os.chmod(wp, 0o755)
    !mkdir -p /content/osrm-backend/build
    !cd /content/osrm-backend/build && CC=/usr/local/bin/cc-nowarn CXX=/usr/local/bin/cxx-nowarn \
        cmake .. -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -6
    !cd /content/osrm-backend/build && cmake --build . -j$(nproc) 2>&1 | tail -15
    !cd /content/osrm-backend/build && cmake --build . --target install 2>&1 | tail -3
    # Cachear binarios + perfiles a Drive para próximas sesiones
    OSRM_CACHE.mkdir(parents=True, exist_ok=True)
    for b in OSRM_BINS:
        p = f"/usr/local/bin/{b}"
        if os.path.exists(p):
            shutil.copy(p, OSRM_CACHE / b)
    shutil.copytree("/content/osrm-backend/profiles", OSRM_CACHE / "profiles", dirs_exist_ok=True)
    shutil.copytree("/content/osrm-backend/profiles", PROFILES, dirs_exist_ok=True)

!ldconfig 2>/dev/null
print("\n=== verificación OSRM ===")
!osrm-extract --version

In [ ]:
# === Paths: env vars apuntando a Drive (QW-5: pipeline/_paths.py) ===
# El pipeline lee IDB_ACCESS_DATA_ROOT / IDB_ACCESS_RESULTS_ROOT vía
# pipeline/_paths.py — sin env vars cae al layout por defecto del repo.
# Los env vars se propagan a los subprocesses (uv run ...) automáticamente.
DATA_DIR = DRIVE / "inputs"
RESULTS_DIR = DRIVE / "outputs"
for p in [DATA_DIR, RESULTS_DIR,
          DATA_DIR / "transportation" / "osrm" / "_pbf",
          RESULTS_DIR / "osrm",
          RESULTS_DIR / "accessibility",
          RESULTS_DIR / "logs"]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["IDB_ACCESS_DATA_ROOT"] = str(DATA_DIR)
os.environ["IDB_ACCESS_RESULTS_ROOT"] = str(RESULTS_DIR)
print("Paths configurados (sin symlinks):")
print(f"  IDB_ACCESS_DATA_ROOT    = {DATA_DIR}")
print(f"  IDB_ACCESS_RESULTS_ROOT = {RESULTS_DIR}")


In [ ]:
# === Validación de inputs (los que de verdad usa 09b/10b) ===
# 09b lee el archivo de escuelas LAC (NO el CIMA por país) + el grid WorldPop.
# 10b agrega con los polígonos BID nivel 2.
required = {
    "Escuelas LAC (09b)": DATA_DIR / "schools" / "AR" / "LAC_schools_k12_with_context.csv",
    "Grid WorldPop (09b)": DATA_DIR / "population" / "WorldPop" / "processed" / f"population_grid_{COUNTRY}.csv",
    "Polígonos BID L2 (10b)": DATA_DIR / "bounderys" / "LAC" / "level 2" / "lac-level-2.csv",
}
missing = {k: str(p) for k, p in required.items() if not p.exists()}
if missing:
    raise FileNotFoundError(
        f"Faltan inputs en Drive para {COUNTRY}:\n  " +
        "\n  ".join(f"{k}: {v}" for k, v in missing.items()) +
        f"\n\nSubílos a {DRIVE_ROOT}/inputs/ — ver README."
    )
print(f"Inputs OK para {COUNTRY}:")
for k, p in required.items():
    print(f"  {k}: {p.stat().st_size/1e6:.1f} MB")

In [ ]:
# === PBF de OSM (cache en Drive entre sesiones) + provenance ===
import json, datetime, subprocess, shutil

PBF_DIR = DATA_DIR / "transportation" / "osrm" / "_pbf"
PBF_DIR.mkdir(parents=True, exist_ok=True)
PBF = PBF_DIR / f"{COUNTRY.lower()}.osm.pbf"
PBF_URL = f"https://download.geofabrik.de/{GEOFABRIK_PATH}-latest.osm.pbf"

if not PBF.exists():
    print(f"Descargando {PBF_URL}...")
    !curl -fL --progress-bar "{PBF_URL}" -o "{PBF}"
else:
    print(f"PBF cacheado: {PBF} ({PBF.stat().st_size/1e6:.0f} MB)")

# Provenance: el timestamp del extract (PROV-2) es nice-to-have. osmium es
# opcional — si no está instalado, registramos 'unknown' y seguimos: no debe
# bloquear el cómputo de las matrices.
pbf_osm_ts = "unknown"
if shutil.which("osmium"):
    info = subprocess.run(
        ["osmium", "fileinfo", "--extended", str(PBF)],
        capture_output=True, text=True,
    )
    pbf_osm_ts = next(
        (line.split(":", 1)[1].strip()
         for line in info.stdout.splitlines()
         if "osmosis_replication_timestamp" in line.lower() or "timestamp" in line.lower()),
        "unknown",
    )
else:
    print("  osmium no encontrado — pbf_osm_timestamp='unknown' (no bloquea)")

prov = {
    "country_iso3": COUNTRY,
    "pbf_url": PBF_URL,
    "pbf_path": str(PBF.relative_to(DATA_DIR)),
    "pbf_size_bytes": PBF.stat().st_size,
    "pbf_osm_timestamp": pbf_osm_ts,
    "downloaded_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "runtime": "google_colab",
}
(PBF_DIR / f"{COUNTRY.lower()}_provenance.json").write_text(json.dumps(prov, indent=2))
print(json.dumps(prov, indent=2))

In [ ]:
# === Build de los grafos OSRM (foot + car) — binarios nativos ===
# Build en disco LOCAL de Colab (/content/osrm, rápido) y cacheo del grafo
# terminado a Drive. Reanudar = copiar de Drive a local (instantáneo).
import shutil
from pathlib import Path

OSRM_DIR = DATA_DIR / "transportation" / "osrm"     # cache en Drive
LOCAL_OSRM = Path("/content/osrm"); LOCAL_OSRM.mkdir(exist_ok=True)
BASE = COUNTRY.lower()

def build_graph(profile_name: str) -> Path:
    drive_dir = OSRM_DIR / f"{COUNTRY}_{profile_name}"
    local_dir = LOCAL_OSRM / f"{COUNTRY}_{profile_name}"
    local_mldgr = local_dir / f"{BASE}.osrm.mldgr"
    drive_mldgr = drive_dir / f"{BASE}.osrm.mldgr"

    if local_mldgr.exists():
        print(f"  [{profile_name}] grafo local presente — skip"); return local_dir
    if drive_mldgr.exists():
        print(f"  [{profile_name}] grafo cacheado en Drive — copiando a local...")
        shutil.copytree(drive_dir, local_dir, dirs_exist_ok=True); return local_dir

    local_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(PBF, local_dir / f"{BASE}.osm.pbf")
    lua = PROFILES / f"{profile_name}.lua"
    print(f"  [{profile_name}] osrm-extract...")
    !osrm-extract -p "{lua}" "{local_dir}/{BASE}.osm.pbf"
    print(f"  [{profile_name}] osrm-partition...")
    !osrm-partition "{local_dir}/{BASE}.osrm"
    print(f"  [{profile_name}] osrm-customize...")
    !osrm-customize "{local_dir}/{BASE}.osrm"

    print(f"  [{profile_name}] cacheando grafo a Drive...")
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
    return local_dir

print("=== build foot ===")
foot_dir = build_graph("foot")
print("=== build car ===")
car_dir = build_graph("car")
print(f"\nGrafos listos (local): {LOCAL_OSRM}")

In [ ]:
# === Matrices walking (foot graph): los 6 combos en UNA pasada (--all-combos) ===
# Servidor osrm-routed nativo (subprocess); el cliente 09b consulta :5000.
# --all-combos rutea cada celda UNA sola vez por modo y deriva los 6 parquets
# (public/private × 3 niveles) de esa única respuesta OSRM → ~6x menos búsquedas
# de red que el loop por-combo anterior. La salida es bit-identica al método
# por-combo (validado en tests/test_osrm_all_combos_equivalence.py).
# Sin --overwrite: los parquets ya escritos se saltean → la corrida reanuda
# donde quedó si la sesión se corta (clave para BRA).
import subprocess, time, socket

OSRM_PORT = 5000

def wait_for_osrm(port: int, timeout: float = 90) -> bool:
    start = time.time()
    while time.time() - start < timeout:
        try:
            s = socket.create_connection(("127.0.0.1", port), timeout=1); s.close(); return True
        except OSError:
            time.sleep(0.5)
    return False

def serve_and_run(graph_dir: Path, mode: str) -> None:
    log_path = RESULTS_DIR / "logs" / f"colab_osrm_{COUNTRY}_{mode}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    osrm_file = graph_dir / f"{BASE}.osrm"
    with open(log_path, "w") as logfh:
        proc = subprocess.Popen(
            ["osrm-routed", "--algorithm", "mld", "-i", "0.0.0.0",
             "-p", str(OSRM_PORT), str(osrm_file)],
            stdout=logfh, stderr=subprocess.STDOUT)
        try:
            assert wait_for_osrm(OSRM_PORT), f"osrm-routed no levantó en :{OSRM_PORT} (ver {log_path})."
            print(f"osrm-routed ({mode}) escuchando :{OSRM_PORT}")
            print(f"\n--- {mode}: 6 combos (public/private × 3 niveles) en una pasada ---")
            !cd {REPO} && uv run python pipeline/09b_travel_time_osrm.py \
                --country {COUNTRY} --mode {mode} --all-combos --port {OSRM_PORT}
        finally:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except Exception:
                proc.kill()
            time.sleep(2)

serve_and_run(foot_dir, "walking")
print("\nwalking listo; osrm-routed detenido")

In [ ]:
# === Matrices motorized (car graph) × 2 sectores × 3 niveles ===
# Reusa serve_and_run() definido en la celda walking.
serve_and_run(car_dir, "motorized")
print("\nmotorized listo; osrm-routed detenido")

In [ ]:
# === Step 10b — agregación SCL (append-by-país, safe gracias a fe1c7da) ===
!cd {REPO} && uv run python pipeline/10b_accessibility_aggregate_osrm.py \
    --countries {COUNTRY}

In [ ]:
# === Verificación del output ===
import pandas as pd

scl_path = RESULTS_DIR / "accessibility" / "accessibility_osrm_scl.csv"
scl = pd.read_csv(scl_path, encoding="utf-8-sig", keep_default_na=False, low_memory=False)
isos = sorted(scl["isoalpha3"].unique())
print(f"Filas totales: {len(scl):,}")
print(f"Países en el CSV: {len(isos)}  →  {isos}")
n_country = (scl["isoalpha3"] == COUNTRY).sum()
print(f"\nFilas para {COUNTRY}: {n_country:,}")
if n_country == 0:
    raise RuntimeError(f"El step 10b no agregó filas para {COUNTRY} — revisar logs.")

# Sanity-check unicidad por la tupla clave
keys = ["isoalpha3", "idgeo", "admin1_pcode", "admin2_pcode",
        "indicator", "mode", "education_level", "age", "sector",
        "area", "quintile", "time_band"]
n_dup = int(scl.duplicated(subset=keys).sum())
assert n_dup == 0, f"Hay {n_dup} filas duplicadas en la tupla clave SCL"
print("\nOK — sin duplicados en la tupla clave.")
print(f"\nOutputs persistidos en Drive: {DRIVE_ROOT}/outputs/accessibility/")


## Listo

El CSV final está en `MyDrive/idb_access/outputs/accessibility/accessibility_osrm_scl.csv` (con sectores Public/Private/Total). Los hasta 12 parquets del país en `MyDrive/idb_access/outputs/osrm/{COUNTRY}_{mode}_{level}_{sector}_osrm.parquet`. El timestamp del extract OSM en `inputs/transportation/osrm/_pbf/{iso}_provenance.json`.

**Siguientes pasos**
1. Volvé a la celda 1 y cambiá `COUNTRY` al siguiente país (MEX, después BRA).
2. `Runtime → Disconnect and delete runtime` para liberar RAM, después `Run all` de nuevo.
3. Una vez los 3 corridos, descargá `accessibility_osrm_scl.csv` y subilo a `results/accessibility/` en tu repo local.

**Troubleshooting** → ver `notebooks/README.md` sección Troubleshooting.